# 01 Phase coherent readout — on QCSS

Adapted from the official QICK demo [`01_Phase_coherent_readout.ipynb`](https://github.com/openquantumhardware/qick/blob/9d98886/qick_demos/01_Phase_coherent_readout.ipynb) (qick 0.2.370, MIT License, © 2021 Open Quantum Hardware).

Calibrate the phase of a phase-coherent readout: measure the readout phase against frequency through loopback cables and fit the cable delay.

**What changes from the official demo:** the two cells marked `QCSS` — starting the simulated board, and connecting through `make_proxy` instead of running on the board. Every other cell is the upstream code, unchanged except where a `QCSS` comment says so.

**What this shows:** qick's own client library compiles, uploads and acquires exactly as it does against a board, and gets back data of the shape, dtype and scale a board returns. It does **not** prove the numbers are identical to any particular physical board: no real-board reference data is compared here.

In [ ]:
# QCSS: start the simulated ZCU216 board (tProc v1 firmware, as these demos expect)
import sys; sys.path.insert(0, '..')
import _lab, _proof
lab = _lab.start(qick=True, firmware='zcu216_v1_5q', loopback=('1:0', '2:1'))

# <center> Calibrating the QICK for phase coherent readout

### <center> In this demo you will calibrate the QICK clocks to have the same phase.

Before you measure a resonance with your QICK this is the first calibration you should do. It is a calibration for the two synthesizers which belong to the QICK signal generator and the QICK readout, respectively. The two synthesizers are running at the same frequency, but there is initially a constant phase difference $\phi$ between these two synthesizers. Doing this calibration results in you finding that phase difference $\phi$. In your subsequent measurements, you can specify this initial phase difference $\phi$ to compensate for it. From then on, the signal generator can synthesize any frequency and then if you read in data (doing a digital down conversion in the process), the readout will still be phase coherent with respect to the signal generator. 

<center> The angular frequency $\omega = 2 \pi f$ .

<center> Also, $\phi = (\omega t) + \phi_0$. 

<center> So, $\phi = (2 \pi f)*t + \phi_0 $. 

If $f$ goes up linearly, the phase difference will also change linearly (it will either increase or decrease, depending on whether the readout is ahead or behind of the signal generator- this is randomly determined each time the board clocks are initialized). Once the phase hits 360 degrees it cycles back to 0 again. For a readout frequency of interest $f_i$ there is a corresponding phase difference $\phi_i$. In this demonstration we assume $f_i \approx 180$ MHz. You can plot $\phi(f)$ and evaluate $\phi(f_i)=\phi_i$.

In [ ]:
# Import the QICK drivers and auxiliary libraries
from qick import *
from qick.parser import load_program
%pylab inline

In [ ]:
# Load bitstream with custom overlay
# QCSS: the official demo runs on the board itself (soc = QickSoc()). From a PC you reach a board through
# QICK's Pyro4 server instead (qick.pyro.make_proxy) -- on a real board only ns_host would differ.
from qick.pyro import make_proxy
soc, soccfg = make_proxy(ns_host="127.0.0.1", ns_port=8888, proxy_name="myqick")
_lab.keep_errors_visible()  # QCSS: make_proxy hides cell errors from Jupyter; keep them failing
# Since we're running locally on the QICK, we don't need a separate QickConfig object.
# If running remotely, you could generate a QickConfig from the QickSoc:
#     soccfg = QickConfig(soc.get_cfg())
# or save the config to file, and load it later:
#     with open("qick_config.json", "w") as f:
#         f.write(soc.dump_cfg())
#     soccfg = QickConfig("qick_config.json")
# print(soccfg)

# Print the QICK configuration
print(soc)

# loopback and monitor channels
out_chs = [1,2]  # QCSS bench cables: generator 1 -> readout 0, generator 2 -> readout 1


### Hardware Configuration

tProc channel 7   : DAC 229 CH3  <-> Readout channel 0 : ADC 224 CH0

The following are optional:

* tProc channel 6   : DAC 229 CH2       <-> Readout channel 1 : ADC 224 CH1
* tProc channel 1   : DAC 228 CH0(P)    <-> oscilloscope, through a DC-block
* PMOD 0_0 (pin 1 on J48 on the ZCU111) <-> oscilloscope



We are going to drive the DAC outputs in periodic mode, so we only need to specify a short constant waveform and they will continue to run forever. This is useful for the phase measurement, but it's good practice to stop the DACs afterwards using soc.reset_gens().

We define the frequency range to be tested, and some helper functions that will process and print the results. For each frequency we will print the measured phase (what we care about), and also the mean magnitude and RMS of the waveform (for debugging - the magnitude should always be much larger than the RMS).

In [ ]:
# Output frequency f0_v.
###################
# Try it yourself ! Change the output frequency.
###################

f0_start=100
f0_step=0.000250
expts=40
# expts=1

f0_v = np.arange(0,expts)*f0_step+f0_start

f0_v = soccfg.adcfreq(f0_v, gen_ch=1, ro_ch=0)  # QCSS: generator 1 is cabled to readout 0

def calculate_phase(d):
    [xi,xq] = d
    x = xi +1j*xq

    # Average to improve calibration.
    xavg = np.mean(x)

    # Calculate calibration phase.
    fi = np.remainder(np.angle(xavg,deg=True)+360,360)
    return [fi, np.abs(xavg), np.std(x)]

def print_results(res):
    print("freq_i = %f MHz, "%(f0) +
         "phi_i = (%.2f, %.2f) deg, " % tuple([res[i][0] for i in range(2)]) +
         "mag = (%.2f, %.2f), " % tuple([res[i][1] for i in range(2)]) +
         "RMS = (%.2f, %.2f) ADU" % tuple([res[i][2] for i in range(2)]))
#     print("freq_i = %f MHz, phi_i = (%.2f, %.2f) deg, mag = (%.2f, %.2f), RMS = (%.2f, %.2f) ADU" %(f0,*out_array,*A,*xrms))

### Measurement methods
We are going to make the same measurement in several ways, to demonstrate different ways of running the QICK (and hopefully gain some understanding of what's going on under the hood). You can jump to the analysis at the end at any point to plot the data - all methods write the results to the same array.

For the first two methods, we will have a common Python method - below - that acquires the data, but doesn't load the tProc program.

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap('QCSS does not serve soc.tproc (raw tProc memory and program access); the AveragerProgram cells below make the same measurement through the served methods', 'userdocs/reference/qick-surface.md')

# def measure_phase_decimated(soc, f0, fi0, out_array):
#     """
#     Measure the phase shift for a given frequency using decimated readout, and write it into the provided array.
#     Assumes that the tProc is already loaded with a program that reads the frequency and phase from memory locations 123 and 124.
#     """
#     readout_length = 1000
# 
#     f0_dac_int = soccfg.freq2reg(f=f0) # Frequency of signal generator
#     
#     soc.tproc.single_write(addr=123, data=f0_dac_int)
# 
#     fi0_int = soccfg.deg2reg(fi0, gen_ch=6)
#     soc.tproc.single_write(addr=124, data=fi0_int)
# 
#     # set readout parameters
#     ro_pars = {'freq': f0, 'gen_ch': out_chs[0]}
#     
#     # Configure readout.
#     for ch in range(2):
#         ro_cfg = soccfg.calc_ro_regs(soccfg['readouts'][ch], phase=0, sel='product')
#         soccfg.calc_ro_freq(soccfg['readouts'][ch], ro_pars, ro_cfg, False, 0)
#         soc.configure_readout(ch=ch, ro_regs=ro_cfg)
# #         soc.config_avg(ch=ch, address=0, length=readout_length)
#         soc.config_buf(ch=ch, address=0, length=readout_length)
#         soc.enable_buf(ch=ch, enable_avg=False, enable_buf=True)
#     
#     # Start tProc.
#     soc.start_tproc()
# 
#     time.sleep(0.1)
#     
#     # Get data.
#     data = [soc.get_decimated(ch=ch, length=readout_length).T for ch in range(2)]
# #     print(data)
# 
#     # Process data.
#     res = [calculate_phase(d) for d in data]
# 
#     # Save results.
#     out_array[:] = [a[0] for a in res]
#     print_results(res)

### Load ASM from text file

First, let's program the tProc from an assembly language file, which needs to be located in the same directory when you run it: `01_phase_calibration.asm`

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap('QCSS does not serve soc.tproc (raw tProc memory and program access); the AveragerProgram cells below make the same measurement through the served methods', 'userdocs/reference/qick-surface.md')

# # Constant, real envelope.
# # The length must be at least 16 times the nsamp parameter passed to the signal generators.
# xg = np.zeros((160,2), dtype=np.int16)
# xg[:,0] = 30000
# 
# # Upload waveform.
# for ch in out_chs:
#     # Set the DAC channels to be in 1st Nyquist zone mode
#     soc.set_nyquist(ch=ch,nqz=1)
#     soc.load_envelope(ch=ch, data=xg, addr=0)
# 
# # Load program.
# load_program(soc, prog="01_phase_calibration.asm")
# 
# # Output phase.
# fi0 = 0
# fi_v = np.zeros((2,len(f0_v)))
# for ii, f0 in enumerate(f0_v):
#     measure_phase_decimated(soc, f0, fi0, fi_v[:,ii])
# 
# soc.reset_gens()

### QickProgram

It's usually easier to define the tProc program dynamically in Python, rather than loading it from a separate file. The low-level tool for this is QickProgram.

If you compare `01_phase_calibration.asm` to the commands below, you can see the resemblance; if you convert the QickProgram to .asm format with print(prog) it is very clear.

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap('QCSS does not serve soc.tproc (raw tProc memory and program access); the AveragerProgram cells below make the same measurement through the served methods', 'userdocs/reference/qick-surface.md')

# nsamp = 10
# 
# prog = QickProgram(soccfg)
# 
# # Constant, real envelope.
# # The length must be at least 16 times the nsamp parameter passed to the signal generators.
# xg = np.zeros((16*nsamp,2), dtype=np.int16)
# xg[:,0] = 30000
# 
# for ch in out_chs:
#     # Set the DAC channels to be in 1st Nyquist zone mode
#     prog.declare_gen(ch=ch,nqz=1)
#     # Upload waveform. Since this is the only waveform in this generator, it will start at address 0.
#     prog.add_pulse(ch=ch, name="const", idata=xg[:,0], qdata=xg[:,1])
# 
# prog.memri(1,1,123,"freq")
# prog.memri(1,2,124,"phase")
# prog.regwi(1,3,32000,"gain")
# prog.regwi(1,4,nsamp,"nsamp: generator will consume 16*nsamp DAC values.")
# prog.regwi(1,5,0x4,"b00100 -> phrst = 0, stdysel = 0, mode = 1, outsel = 00")
# prog.bitwi(1,5,5,"<<",16)
# prog.bitw(1,4,4,"|",5)
# prog.regwi(1,6,0,"start time")
# prog.synci(1000)
# prog.comment("Set trigger.")
# prog.regwi(0,1,0x1,"Send a pulse on pmod 0_0 (pin 1 on J48 on the ZCU111).")
# prog.seti(soccfg['readouts'][0]['trigger_port'],0,1,0,"Start the pulse.")
# prog.seti(soccfg['readouts'][0]['trigger_port'],0,0,100,"End the pulse after 100 ticks (260 ns).")
# 
# prog.comment("Program signal generators in periodic mode.")
# prog.comment("Channels 4, 5, 6, 7 -> DAC 229 CH0/1/2/3.")
# for ch in out_chs:
#     # tuse the tProc channel number for the gen
#     prog.set(soccfg['gens'][ch]['tproc_ch'],1,1,2,0,3,4,6)
# prog.synci(1000)
# prog.comment("Set trigger.")
# prog.regwi(0,1,0xc000,"Trigger both buffers.")
# prog.seti(soccfg['readouts'][0]['trigger_port'],0,1,0,"Start the pulse.")
# prog.seti(soccfg['readouts'][0]['trigger_port'],0,0,100,"End the pulse after 100 ticks (260 ns).")
# prog.end()
# 
# 
# 
# print(prog)
# 
# prog.config_all(soc)
# 
# 
# 
# # Output phase.
# fi0 = 0
# 
# fi_v = np.zeros((2,len(f0_v)))
# 
# # Load program.
# for ii, f0 in enumerate(f0_v):
#     measure_phase_decimated(soc, f0, fi0, fi_v[:,ii])
# 
# soc.reset_gens()

### AveragerProgram
Instead of writing your acquisition code from scratch every time, it is usually easier to use AveragerProgram, which combines the QickProgram functionality with a set of built-in acquisition methods. AveragerProgram can also use the tProc to run the acquisition in a loop, which is much faster.

In [ ]:
class SingleFreqProgram(AveragerProgram):
    def __init__(self,soccfg, cfg):
        super().__init__(soccfg, cfg)

    def initialize(self):
        cfg=self.cfg   
        
        # configure the readout lengths and downconversion frequencies
        for ch in range(2):
            self.declare_readout(ch=ch, length=self.cfg["readout_length"],
                                 freq=self.cfg["pulse_freq"])

        idata = 30000*np.ones(16*cfg["length"])

        for ch in self.cfg['out_ch']:
            self.declare_gen(ch=ch, nqz=1)
            self.add_pulse(ch=ch, name="measure", idata=idata)
        
        freq=soccfg.freq2reg(cfg["pulse_freq"])  # convert frequency to dac frequency
        self.trigger(pins=[0], t=0) # send a pulse on pmod0_0, for scope trigger
        for ch in self.cfg['out_ch']:
            self.set_pulse_registers(ch=ch, style="arb", freq=freq, phase=cfg["res_phase"], gain=cfg["pulse_gain"], 
                                     waveform="measure", mode="periodic")

        self.synci(200)  # give processor some time to configure pulses
    
    def body(self):
        self.trigger(adcs=[0,1],adc_trig_offset=self.cfg["adc_trig_offset"])  # trigger the adc acquisition
        for ch in self.cfg['out_ch']:
            self.pulse(ch=ch, t=0) # play readout pulse
        self.wait_all() # control should wait until the readout is over
        self.sync_all(200)  # wait for measurement to complete

config={"out_ch":out_chs,
        "reps":1, # --Fixed
        "res_phase":soccfg.deg2reg(0), # --Fixed
        
        "length":10, # [Clock ticks]
        # Try varying length from 10-100 clock ticks
        
        "readout_length":1000, # [Clock ticks]
        # Try varying readout_length from 50-1000 clock ticks

        "pulse_gain":32000, # [DAC units]
        # Try varying pulse_gain from 500 to 30000 DAC units

        "pulse_freq": 100, # [MHz]
        # In this program the signal is up and downconverted digitally so you won't see any frequency
        # components in the I/Q traces below. But since the signal gain depends on frequency, 
        # if you lower pulse_freq you will see an increased gain.

        "adc_trig_offset": 1000, # [Clock ticks]
        # Try varying adc_trig_offset from 100 to 220 clock ticks

        "soft_avgs":1
        # Try varying soft_avgs from 1 to 200 averages

       }

### Decimated readout
First, let's run the AveragerProgram in the same mode that we were using before: decimated readout, where we save the full waveform. If you increase the value of `soft_avgs`, the AveragerProgram will average multiple waveforms before giving you the results, so the measurement will take longer (past 10, the readout time is noticeable) but the RMS of the waveform will decrease.

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap("Known QCSS bug: a generator in mode='periodic' should repeat its waveform until the next pulse, but QCSS plays it once, so the readout 1000 ticks later hears only noise. Fix pending; this cell and the analysis that depends on it stay skipped until then", 'src/qcss/boards/qick/gens.py (Pulse.periodic is decoded but never used)')

# res=[]
# config['reps'] = 1
# 
# # change this from 1 to 50
# config['soft_avgs'] = 1
# 
# # for f0 in [101]:
# for f0 in f0_v:
#     config['pulse_freq'] = f0
#     prog =SingleFreqProgram(soccfg, config)
# #     print(prog)
#     data = prog.acquire_decimated(soc, progress=False)
# #     print(data)
#     res.append([calculate_phase(d) for d in data])
#     print_results(res[-1])
# fi_v=np.array([[a[0] for a in r] for r in res]).T
# 
# soc.reset_gens()

### Accumulated readout
Since the demodulated waveform is constant over the acquisition window, we don't really need to save the full waveform. In other words, we should use the QICK's accumulating buffer (which sums the values in the window) instead of the decimated buffer.

Since we now get one number per acquisition, the RMS is now calculated over acquisitions, not over waveform samples.

You should see that a "repetition" takes much less time than a "soft average." Normally, decimated readout is only used for finding the correct window to accumulate over, or for debugging.

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap("Known QCSS bug: a generator in mode='periodic' should repeat its waveform until the next pulse, but QCSS plays it once, so the readout 1000 ticks later hears only noise. Fix pending; this cell and the analysis that depends on it stay skipped until then", 'src/qcss/boards/qick/gens.py (Pulse.periodic is decoded but never used)')

# # change this from 10 to 1000
# config['reps'] = 1000
# 
# config['soft_avgs'] = 1
# res=[]
# for f0 in f0_v:
#     config['pulse_freq'] = f0
#     prog =SingleFreqProgram(soccfg, config)
#     avg_data = prog.acquire(soc, progress=False)
#     data = [[prog.di_buf[i]/config['readout_length'], prog.dq_buf[i]/config['readout_length']] for i in range(2)]
#     res.append([calculate_phase(d) for d in data])
#     print_results(res[-1])
# fi_v=np.array([[a[0] for a in r] for r in res]).T
# soc.reset_gens()

### Summary, and RAveragerProgram
This mode (AveragerProgram with accumulated readout) is one of the typical ways that we run the QICK.

The other way uses RAveragerProgram, which has a tProc program with two nested loops: an inner loop for repetitions (just like AveragerProgram) and an outer loop for sweeping a parameter. Because Python can no longer be the bottleneck limiting the measurement speed, this is often much faster.

However, RAveragerProgram cannot be used for this particular measurement because we need to sweep the frequency of both the DAC and the ADC simultaneously - the tProc can control the DAC frequency, but not the ADC frequency. 

### Analysis
Now we can plot the phase as a function of frequency.

As explained at the beginning, this data can be represented as the sum of a phase offset and a time delay. Over a small frequency range, this looks like a linear dependence on frequency.

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap("Known QCSS bug: a generator in mode='periodic' should repeat its waveform until the next pulse, but QCSS plays it once, so the readout 1000 ticks later hears only noise. Fix pending; this cell and the analysis that depends on it stay skipped until then", 'src/qcss/boards/qick/gens.py (Pulse.periodic is decoded but never used)')

# # Plot results.
# # plt.figure(2)
# fig, axs = plt.subplots(2,1)
# 
# for ch in range(2):
#     plot = axs[ch]
#     plot.plot(f0_v,fi_v[ch])
#     plot.plot(f0_v,fi_v[ch], marker='.', linestyle="None",color="Red")
#     plot.set_title(r"$\phi$ vs $f$")
#     plot.set_ylabel(r"$\phi$ (degrees)")
#     plot.set_xlabel(r"$f$ (MHz)")
# 
# fig.savefig("images/Phase_calibration.pdf", dpi=350)

For example, if your cavity tone was generated with an IF of 420.9 MHz, you would look at the above table and see that <code> freq_i </code> = 420.9 MHz corresponds to <code> phi_i </code> = 308.7 degrees (that's specific to this board session, for your QICK it will be different). That value of <code> phi_i </code> will be used as the value of the <code> config </code> file parameter associated with the offset phase of the cavity pulse (the parameter is called <code> res_phase </code> in the demo 00_Send_recieve_pulse).

We can try a fit:

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap("Known QCSS bug: a generator in mode='periodic' should repeat its waveform until the next pulse, but QCSS plays it once, so the readout 1000 ticks later hears only noise. Fix pending; this cell and the analysis that depends on it stay skipped until then", 'src/qcss/boards/qick/gens.py (Pulse.periodic is decoded but never used)')

# def phase_residuals(data,prediction):
#     r = np.remainder(data-prediction+180,360)-180
#     return r
#     
# def phase_model(x, f0):
#     return np.remainder(x[0] - 360*x[1]*(f0), 360)
# 
# def phase_func(x, arg):
#     resid = phase_residuals(arg, phase_model(x, f0_v))
#     return resid
# 
# from scipy.optimize import least_squares
# fig, axs = plt.subplots(2,2)
# for ch in range(2):
# 
#     slopes = -1*(fi_v[ch,1:]-fi_v[ch,:-1])/(360*(f0_v[1:]-f0_v[:-1]))
# #     print(slopes)
#     x0 = np.zeros(2)
#     x0[1] = np.median(slopes)
#     x0[0] = np.remainder(np.median(phase_residuals(fi_v[ch],phase_model(x0, f0_v))),360)
#     print("initial estimate: %.2f deg shift, %.2f us delay"% tuple(x0))
# 
#     fit = least_squares(phase_func, x0, args=(fi_v[ch],))
# #     fit = least_squares(phase_func, x0, args=(fi_v[ch]), method='lm', x_scale='jac')
# #     fit = least_squares(phase_func, x0, args=(fi_v[ch]), method='lm', x_scale=(1,1e-5))
#     fit.x[0] = np.remainder(fit.x[0],360)
# #     print(fit.status)
#     print("after minimization: %.2f deg shift, %.2f us delay"% tuple(fit.x))
# 
#     plot = axs[0,ch]
#     plot.set_title(r"$\phi$ vs $f$")
#     plot.set_ylabel(r"$\phi$ (degrees)")
#     plot.set_xlabel(r"$f$ (MHz)")
#     plot.plot(f0_v,fi_v[ch], marker='.', linestyle="None",color="Red")
#     plot.plot(f0_v, phase_model(fit.x, f0_v))
# 
#     plot = axs[1,ch]
#     plot.set_ylabel(r"residual (degrees)")
#     plot.set_xlabel(r"$f$ (MHz)")
#     
#     plot.plot(f0_v, phase_func(fit.x, fi_v[ch]), marker='.', linestyle="None",color="Red")
# #     plot.plot(f0_v, phase_func(x0, (fi_v[ch])), marker='.', linestyle="None",color="Red")

In [ ]:
# QCSS gap: this upstream cell is not run on QCSS; its code is kept below, commented out.
_proof.gap('QCSS does not serve soc.readouts / soc.avg_bufs (driver objects on the board)', 'userdocs/reference/qick-surface.md')

# # "product" (0) and "dds" (1) work as advertised; "input" (2) seems identical to "product" and 3 to "dds"
# #     soc.readouts[0].outsel_reg = 0
# 
# #nsamp=1 or 2 fails in a strange way: 
# #the signal generator plays the entire waveform memory repeatedly for 10-20 seconds
# #during this time, new "set" commands don't work
# 
# 
# # is there ever a reason not to enable both avg and buf?
# #         soc.avg_bufs[ch].config_buf(address=0, length=readout_length)
# #         soc.avg_bufs[ch].enable(avg=False, buf=True)
# 
# 
# 
# #         print(soc.get_accumulated(ch=ch, length=1).T/readout_length, xavg)

## Does the client think it is a real board?

In [ ]:
_proof.qick_surface(soc, soccfg)

## Teardown

In [ ]:
lab.stop()